# Acronyms

This notebook checks whether acronym entries are currently represented in the live ClimateKG instance and how they are linked to the report hierarchy.

We validate the live graph before interpreting any results, following the same query-first pattern as the other notebooks in this series.

**Query UI**: https://climatekg.tibwiki.io/query/
**SPARQL endpoint**: https://climatekg.tibwiki.io/query/proxy/sparql

In [1]:
import os
import pandas as pd
from SPARQLWrapper import JSON
from IPython.display import display, Markdown

from wikibase_auth import DEFAULT_SPARQL_ENDPOINT, DEFAULT_WIKIBASE_URL, build_sparql_client

os.environ.setdefault("CLIMATEKG_SPARQL_USERNAME", "ckg")
os.environ.setdefault("CLIMATEKG_SPARQL_PASSWORD", "fairdata")

SPARQL_ENDPOINT = os.getenv("CLIMATEKG_SPARQL_ENDPOINT", DEFAULT_SPARQL_ENDPOINT)
WIKIBASE_URL = os.getenv("CLIMATEKG_WIKIBASE_URL", DEFAULT_WIKIBASE_URL)

print(f"Endpoint: {SPARQL_ENDPOINT}")
print("Libraries imported successfully")

'ckg'

'fairdata'

Endpoint: https://climatekg.tibwiki.io/query/proxy/sparql
Libraries imported successfully


In [2]:
query = '''
PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?item ?itemLabel ?class ?classLabel ?report ?reportLabel
WHERE {
  ?item wdt:P1 ?class .
  ?class rdfs:label ?classLabel .
  FILTER(LANG(?classLabel) = "en" && ( ?classLabel = "Acronym" || ?classLabel = "Glossary term" || ?classLabel = "Category" ))
  OPTIONAL { ?item wdt:P3 ?report . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?classLabel ?itemLabel
LIMIT 50
'''

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
rows = results.get("results", {}).get("bindings", [])
df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in rows])

print(f"Rows returned: {len(df)}")
display(df.head(20))

Rows returned: 0


""


## Interpretation

If this notebook returns zero rows, that is also a valid result: it means the current live ClimateKG instance does not yet expose acronym or glossary entries under the expected class labels.

This is the important query-first check. Once the graph contains acronym or glossary items, the same pattern can be extended to count entries per report, summarise term families, and compare label coverage across sections and chapters.